# 12 — VADER vs FinBERT: il finance-tuning paga? (backlog D1)

**Contesto**: ADR-023 ha scelto VADER (Layer 1, zero dipendenze) e ha subordinato l'upgrade a FinBERT all'**evidenza empirica**. Il notebook 06 (n=143) non trovò segnale lead col Layer 1. Oggi l'archivio ha ~7k titoli rilevanti per BTC/ETH: è il momento del confronto diretto.

**Setup**: 6.936 titoli (googlenews_btc/eth + Cointelegraph + CoinDesk) scorati con entrambi: VADER compound (già in archivio) e FinBERT (ProsusAI/finbert, P(pos)−P(neg), stesso range [−1,1]). Prezzi daily BTC/ETH dagli ultimi 365 giorni (serie committate). FinBERT gira come **tooling d'esperimento** (venv, non in pyproject): l'eventuale adozione passerebbe da un ADR.

## Ipotesi — scritte PRIMA di vedere i numeri

- **H1 (disaccordo materiale)**: VADER è general-domain: mi aspetto accordo di polarità **< 80%** sui titoli finanziari, con disaccordi sistematici su lessico di settore ("plunge", "outflows", "beats estimates", "hawkish").
- **H2 (allineamento same-day migliore)**: il sentiment giornaliero FinBERT dovrebbe correlare coi return **contemporanei** più di VADER (capisce il linguaggio finanziario, quindi descrive meglio la giornata).
- **H3 (nessun lead, per entrambi)**: coerente con nb 06 ed efficienza weak-form: sentiment(t) → return(t+1) ≈ 0 per entrambi gli scorer.

## Regola di decisione — pre-registrata
FinBERT entra in pipeline (nuovo ADR, dipendenze pesanti nel cron) **solo se**: la correlazione same-day migliora di **≥ +50% relativo su BTC ed ETH insieme**, *oppure* emerge un segnale lead assente in VADER. Altrimenti: si resta su VADER e si documenta (anche un no è un risultato).

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

SCORES = Path('../data/cache/finbert_scores.parquet')
assert SCORES.exists(), (
    'Cache mancante: uv pip install torch transformers, poi '
    'uv run --no-sync python notebooks/score_finbert_cache.py (dalla repo root).'
)
df = pd.read_parquet(SCORES)
ms = json.load(open('../public/data/market_series.json'))
px = {s['symbol']: pd.Series([p['v'] for p in s['points']],
      index=pd.DatetimeIndex(pd.to_datetime([p['t'] for p in s['points']]), tz='UTC'))
      for s in ms['series'] if s['symbol'] in ('BTC', 'ETH')}
ret = {k: v.pct_change().dropna() for k, v in px.items()}
print(f'{len(df)} titoli scorati | prezzi: '
      f"BTC {px['BTC'].index.min().date()}->{px['BTC'].index.max().date()}")
df[['vader', 'finbert']].describe().round(3)

6971 titoli scorati | prezzi: BTC 2025-07-04->2026-07-03


,vader,finbert
count,6971.000,6971.000
mean,-0.033,-0.126
std,0.325,0.527
min,-0.948,-0.968
25%,-0.273,-0.628
50%,0.000,0.006
75%,0.067,0.131
max,0.952,0.939


## 1. H1 — Quanto (e dove) i due scorer sono in disaccordo
Accordo di polarità (soglia neutrale ±0.05), correlazione fra gli score, ed esempi dei disaccordi più estremi — per vedere *chi* dei due legge meglio i titoli finanziari.

In [2]:
def pol(s, eps=0.05):
    return np.where(s > eps, 1, np.where(s < -eps, -1, 0))

pv, pf = pol(df['vader']), pol(df['finbert'])
agree = (pv == pf).mean()
corr_p = df['vader'].corr(df['finbert'])
corr_s = df['vader'].corr(df['finbert'], method='spearman')
print(f'accordo di polarità: {agree:.1%}   pearson={corr_p:.3f}   spearman={corr_s:.3f}')

opposite = df[(pv == 1) & (pf == -1) | (pv == -1) & (pf == 1)]
print(f'polarità OPPOSTE: {len(opposite)} titoli ({len(opposite)/len(df):.1%})')

d = (df['finbert'] - df['vader']).abs().sort_values(ascending=False)
print('\n--- disaccordi più estremi (titolo | vader | finbert) ---')
for i in d.index[:10]:
    r = df.loc[i]
    print(f"  {r['vader']:+.2f} | {r['finbert']:+.2f} | {r['title'][:90]}")

accordo di polarità: 49.7%   pearson=0.381   spearman=0.405
polarità OPPOSTE: 808 titoli (11.6%)

--- disaccordi più estremi (titolo | vader | finbert) ---
  +0.84 | -0.90 | Bitcoin price news: BTC again lower as traditional markets gain on report of imminent peac
  -0.83 | +0.89 | Crypto News Today (June 8): BTC Back Above $63K as the War Between Justin Sun War and Trum
  +0.84 | -0.80 | Bitcoin drops below $63,000 as global risk assets sell off, erasing gains from US-Iran pea
  +0.69 | -0.94 | Ethereum Price Forecast: Declining active addresses weigh on ETH as open interest climbs t
  +0.65 | -0.96 | Anthropic's pre-IPO shares fall as U.S. government shuts down its most powerful AI model
  -0.77 | +0.84 | Strategy Inc. rebounds with $54B in Bitcoin, surpassing debt by $48B since 2022 crisis. - 
  +0.69 | -0.90 | Ethereum open interest drops 25% as $1,500 support comes into focus - TradingView
  -0.71 | +0.84 | Bitcoin's 'fear gauge' surges nearly 20%, its biggest jump since Feb. 5 cr

## 2. H2 — Allineamento coi return contemporanei
Sentiment medio giornaliero per asset (fonte dedicata + newswire) vs return dello stesso giorno UTC. Correlazione di Spearman (robusta agli outlier), su giorni con ≥ 3 titoli.

In [3]:
ASSET_SOURCES = {'BTC': ['googlenews_btc', 'cointelegraph', 'coindesk'],
                 'ETH': ['googlenews_eth', 'cointelegraph', 'coindesk']}
MIN_TITLES = 3

def daily_sentiment(sym, col):
    sub = df[df['source'].isin(ASSET_SOURCES[sym])]
    day = pd.DatetimeIndex(sub.index).normalize()
    g = sub.groupby(day)[col].agg(['mean', 'count'])
    return g[g['count'] >= MIN_TITLES]['mean']

rows = []
for sym in ('BTC', 'ETH'):
    r = ret[sym]
    for col in ('vader', 'finbert'):
        s = daily_sentiment(sym, col)
        j = pd.concat([s, r], axis=1, join='inner').dropna()
        j.columns = ['sent', 'ret']
        rho, p = stats.spearmanr(j['sent'], j['ret'])
        rows.append({'asset': sym, 'scorer': col, 'giorni': len(j),
                     'spearman_same_day': round(rho, 3), 'p_value': round(p, 4)})
same_day = pd.DataFrame(rows).set_index(['asset', 'scorer'])
same_day

giorni  spearman_same_day  p_value
asset scorer                                     
BTC   vader        39              0.379   0.0173
      finbert      39              0.517   0.0008
ETH   vader        45              0.256   0.0900
      finbert      45              0.476   0.0009

## 3. H3 — Lead (t → t+1) e reverse (il prezzo guida le news?)
`sent(t) vs ret(t+1)`: potere predittivo. `ret(t) vs sent(t+1)`: la stampa che insegue il prezzo (atteso dominante, da nb 06).

In [4]:
rows = []
for sym in ('BTC', 'ETH'):
    r = ret[sym]
    for col in ('vader', 'finbert'):
        s = daily_sentiment(sym, col)
        j = pd.concat([s.rename('sent'), r.rename('ret')], axis=1, sort=True).dropna()
        lead = stats.spearmanr(j['sent'].iloc[:-1], j['ret'].iloc[1:])
        rev = stats.spearmanr(j['ret'].iloc[:-1], j['sent'].iloc[1:])
        rows.append({'asset': sym, 'scorer': col,
                     'lead sent(t)->ret(t+1)': round(lead.statistic, 3),
                     'lead p': round(lead.pvalue, 3),
                     'reverse ret(t)->sent(t+1)': round(rev.statistic, 3),
                     'reverse p': round(rev.pvalue, 3)})
leads = pd.DataFrame(rows).set_index(['asset', 'scorer'])
leads

lead sent(t)->ret(t+1)  lead p  reverse ret(t)->sent(t+1)  \
asset scorer                                                               
BTC   vader                    -0.029   0.865                      0.494   
      finbert                   0.111   0.508                      0.576   
ETH   vader                    -0.104   0.500                      0.322   
      finbert                   0.088   0.570                      0.423   

               reverse p  
asset scorer              
BTC   vader        0.002  
      finbert      0.000  
ETH   vader        0.033  
      finbert      0.004

## 4. Verdetto — contro la regola pre-registrata

**H1 (disaccordo materiale) — CONFERMATA, oltre le attese.** Accordo di polarità **49.7%** (previsto <80%): i due scorer leggono *metà* dei titoli in modo diverso; l'11.6% ha polarità **opposte**. I disaccordi estremi mostrano il pattern atteso: FinBERT capisce "BTC lower as markets gain on peace report" (negativo; VADER +0.84, ingannato da *gain/peace*) e "declining active addresses weigh on ETH" (−0.94; VADER +0.69). Non è però infallibile: "Bitcoin's fear gauge surges" lo legge positivo (+0.84) — *surges* fuori contesto.

**H2 (allineamento same-day migliore) — confermata direzionalmente, ma la regola di adozione NON è soddisfatta.**

| Asset | VADER ρ (p) | FinBERT ρ (p) | Δ relativo |
|---|---|---|---|
| BTC (n=39gg) | 0.379 (0.017) | **0.517** (0.0008) | **+36%** |
| ETH (n=45gg) | 0.256 (0.090, n.s.) | **0.476** (0.0009) | **+86%** |

FinBERT vince su entrambi — su ETH trasforma una correlazione *non significativa* in fortemente significativa. Ma la regola pre-registrata chiedeva **≥ +50% relativo su BTC ED ETH**: BTC si ferma a +36%. Inoltre su n≈40 giorni la differenza BTC (0.517 vs 0.379) è dentro il rumore campionario di due correlazioni dipendenti.

**H3 (nessun lead) — CONFERMATA.** Lead sent(t)→ret(t+1) tutti insignificanti (|ρ|≤0.11, p≥0.5) per entrambi gli scorer. Il canale dominante è il **reverse**: ret(t)→sent(t+1) con ρ 0.32–0.58 e p≤0.03 — la stampa insegue il prezzo, come nel notebook 06. Nemmeno il finance-tuning sblocca potere predittivo.

### Decisione
**Si resta su VADER in pipeline.** La barra pre-registrata non è stata superata, e i pali non si spostano dopo aver visto i numeri — è l'intero senso della pre-registrazione. Questo è però il **primo risultato concreto pro-FinBERT** del progetto (contemporaneo, non predittivo): descrive meglio la giornata, non la anticipa.

### Follow-up registrato
Rieseguire questo notebook quando il campione giornaliero raddoppia (n≈80–90 giorni per asset, indicativamente fine agosto 2026, col cron a 3h che accumula): se FinBERT supera la barra su entrambi gli asset con n maggiore, si apre l'ADR di adozione (Layer 1.5: FinBERT per lo score dei titoli nel cron, con costo di dipendenze da valutare in quell'ADR).

### Caveat
- n piccolo (39–45 giorni con ≥3 titoli): correlazioni con CI larghi
- allineamento same-day UTC senza lag: misura *descrittiva*, non tradabile (ADR-024 impone lag 1g per uso predittivo — e lì non c'è nulla)
- survivorship dei feed: storia densa solo da fine maggio 2026